# vcc_2026_4: transformer with PIE evidence

Goal: a transformer submission to the 2026 leaderboard before it closes (rules: "until October 22, 2026", two entries per
UTC day; planned cutoff 21 Oct 23:59 UTC). Runs on **vcc2027** (L4 24 GB). The logic lives in `scripts/` and
`code/transformer/`; cells call them through `run` (short steps) and `bg` / `status` / `wait` (long steps in tmux, logs in
`~/vcc/logs/vcc_2026_4/`).

| Phase | What | Where |
|---|---|---|
| 0 | GPU VM, driver, leaderboard rules | vcc2027 |
| A | setup, data copy, provenance checks, data inventory | vcc2027 |
| B | baselines v2 (shared gene set S) | CPU |
| C | PIE: install, control-only contexts, K562 benchmark, PIE tables | GPU |
| D | transformer: loader, model, losses, tests, smoke test | GPU |
| E | dev run, effect-space and H1 pipeline validation | GPU |
| F | final-mode retrain, A-C predictions, generator, `vcc prep` | GPU |

**Fixed decisions (effects_v4):** on-target lfc <= -0.5, MIN_CELLS 30, SE delta method floored at 0.05, unmeasured = NaN;
H1's QC-filtered table is training/validation data, the scoring harness keeps the unfiltered H1 cells. PIE is leaked for
HCT116, HEK293T and H1 (never used there); clean for K562, RPE1, HepG2, Jurkat.

In [ ]:
import os, sys, shutil, subprocess, shlex, time, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
V = Path.home() / 'vcc'
ENV = dict(CODE=str(V), DATA=str(V / 'data'), LIB=str(V / 'lib'), LOGS=str(V / 'logs' / 'vcc_2026_4'),
           RES=str(V / 'results' / 'vcc_2026_4'), B='gs://vcc2026', PY=sys.executable, PIE=str(Path.home() / 'pie'))
os.environ.update(ENV)
os.environ['PATH'] = str(Path.home() / '.local/bin') + os.pathsep + os.environ['PATH']
for d in ('LOGS', 'RES'):
    os.makedirs(ENV[d], exist_ok=True)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)

def run(cmd):
    """Short step: run now (from ~/vcc), stream the output here, stop the cell on error."""
    p = subprocess.Popen(['bash', '-c', 'set -eo pipefail; cd "$CODE"\n' + cmd], env=os.environ,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}) - see the output above')

def bg(name, cmd):
    """Long step: run in tmux session `name`; output goes to ~/vcc/logs/vcc_2026_4/<name>.log."""
    log, sh = Path(ENV['LOGS']) / f'{name}.log', Path(ENV['LOGS']) / f'{name}.sh'
    exports = ''.join(f'export {k}={shlex.quote(v)}\n' for k, v in ENV.items())
    sh.write_text(f'#!/bin/bash\n{exports}export PATH={shlex.quote(os.environ["PATH"])}\nexec > {shlex.quote(str(log))} 2>&1\n'
                  f'trap \'echo "EXIT=$?"\' EXIT\nset -eo pipefail\ncd "$CODE"\n{cmd}\n')
    subprocess.run(['tmux', 'kill-session', '-t', name], stderr=subprocess.DEVNULL)
    subprocess.run(['tmux', 'new-session', '-d', '-s', name, 'bash', str(sh)], check=True)
    print(f'started "{name}" in the background. Check with: status("{name}")')

def status(name, n=20):
    alive = subprocess.run(['tmux', 'has-session', '-t', name], stderr=subprocess.DEVNULL).returncode == 0
    log = Path(ENV['LOGS']) / f'{name}.log'
    lines = log.read_text().splitlines() if log.exists() else []
    state = 'RUNNING' if alive else ('FINISHED OK' if lines and lines[-1] == 'EXIT=0' else 'STOPPED - read the log below')
    print(f'== {name}: {state}\n' + '\n'.join(lines[-n:]))
    return state

def wait(name, every=60):
    """Block until tmux session `name` ends (so the notebook runs top to bottom); raise if it failed."""
    while subprocess.run(['tmux', 'has-session', '-t', name], stderr=subprocess.DEVNULL).returncode == 0:
        time.sleep(every)
    if status(name, n=5) != 'FINISHED OK':
        raise RuntimeError(f'{name} failed - see {ENV["LOGS"]}/{name}.log')

print('python:', sys.executable)
print('tmux:', shutil.which('tmux') or 'MISSING')
run('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || echo "no GPU here"')

## Phase 0: vcc2027

- vcc2027: us-central1-c, g2-standard-8 (8 vCPU, 31 GB RAM), 1x NVIDIA L4 (23 GB), 300 GB pd-balanced, Ubuntu 26.04 minimal.
- Driver: Google's `cuda_installer.pyz` (repo mode) installs NVIDIA 615, whose DKMS module does not build on kernel
  7.0.0-1014-gcp (`dma_buf_move_notify` undefined). Replaced by Ubuntu's prebuilt, signed modules:
  `linux-modules-nvidia-595-server-open-gcp` + `nvidia-headless-595-server-open` (driver 595.91.07, CUDA 13.2).
- Leaderboard: entries "until October 22, 2026", at most twice per day, days start at midnight UTC; no clock time given,
  so plan for 21 Oct 23:59 UTC. Final entries: test lines released 22 Oct, due 5 Nov 23:59 UTC.

In [ ]:
run('nvidia-smi; swapon --show; free -g; df -h /')

## Phase A: setup, data, provenance, inventory

- Python 3.12 venv (`~/venv`) with the vcc2026 packages and torch 2.14.1+cu130; flash and memory-efficient SDPA kernels run
  in bf16 on a 5,000-token batch (flash 1.4 ms, math 29 ms).
- Data come from the bucket (`scripts/pull_from_bucket.sh`, every file's size checked; bucket copies kept).
- Provenance (`scripts/check_provenance.py`, refuses to continue on failure): every effects_v4 table has se_floor 0.05 and
  fdr 0.05 in `uns`; it was built from `lib/qc_cells/` (cell sources) or its source table (Orion pseudobulk, Marson CD4 T);
  on-target -0.5 and MIN_CELLS 30 come from the `perturbation_qc.json` next to the knockdown_qc that `uns` names (the
  tables themselves do not record them); rows = kept knockdowns; the old `lib/effects/` is not used by any v4 code.

In [ ]:
run('bash scripts/pull_from_bucket.sh 2>&1 | grep -E "^ok|MISMATCH|ALL SIZES"')

In [ ]:
run('"$PY" scripts/check_provenance.py')

### Data inventory
Roles: **target** = trained and validated on (validation mode); **validation** = H1, a target only in final mode;
**evidence-only** = CD4 T. PIE: **clean** lines get PIE tables; **leaked** lines (in PIE's training/validation) never do.

In [ ]:
run('"$PY" scripts/data_inventory.py > /dev/null')
display(pd.read_csv(V / 'results/vcc_2026_4/data_inventory.csv'))
display(pd.read_csv(V / 'results/vcc_2026_4/challenge_neighbours.csv'))

## Phase B: baselines v2

S = the 6,013 panel genes measured by every effects_v4 source. Per control cell, library_S = counts over S and
CPM_S = counts / library_S x 1e6. Per gene: log1p(mean CPM_S), detection rate, percentile rank among the S genes, in_S,
measured (`lib/effects_v4/*_baseline_v2.csv`; older baselines untouched; lfc / se not recomputed). CD4 T has no cells:
its pseudobulk control CPM is rescaled over S; detection rate NaN with `detection_missing = 1`. Orion uses its per-cell
controls. Fallback if B is late for D: detection rate + percentile rank from the panel-only CPMs, no log CPM.

In [ ]:
if not all((V / f'lib/effects_v4/{s}_baseline_v2.csv').exists() for s in
           ['replogle_k562_gw', 'arc_h1', 'orion_hct116', 'marson_cd4_stim48hr', 'challenge_A', 'challenge_C']):
    bg('baseline_v2', '"$PY" scripts/build_baseline_v2.py'); wait('baseline_v2')
b2 = json.loads((V / 'lib/effects_v4/baseline_v2.json').read_text())
print('S genes:', b2.pop('_S_genes'))
display(pd.DataFrame(b2).T[['control_cells', 'library_S_genes', 'median_library_S', 'median_ratio_v2_over_panel',
                            'detection_rate_present', 'measured_genes']])

## Phase C: PIE

PIE (Arc, `replogle_xdataset` checkpoint, best_auprc) in its own env (`~/pie`, uv, Python 3.12, torch 2.10 cu128), with the
pinned sources downloaded on first use. Control-only contexts (`scripts/pie_contexts.py`): the line's control cells,
log1p CP10k, genes = the panel genes the line measured that PIE's gene_text has under the same symbol (18,262 for A-C).
K562, RPE1, HepG2, Jurkat use PIE's own context names (k562, rpe1, hepg2, jurkat) so its context_text applies; A-C have
no context_text (masked). `scripts/pie_run.sh` = `pie prep` + `pie infer` (timed); `scripts/pie_tables.py` writes
`lib/pie_tables/pie_<line>.h5ad` (layers lfc_pred, p_de; NaN where PIE gave no prediction). PIE is leaked for HCT116,
HEK293T and H1: no PIE context or table is ever made for them (the scripts refuse).

*Result: filled in when the phase runs.*

In [ ]:
PIE_LINES = ['A', 'K562', 'RPE1', 'HepG2', 'Jurkat', 'B', 'C']
if not all((V / f'lib/pie/contexts/{l}.h5ad').exists() for l in PIE_LINES):
    bg('pie_contexts', '"$PY" scripts/pie_contexts.py --lines ' + ' '.join(PIE_LINES)); wait('pie_contexts')
for line in PIE_LINES:                                    # line A first: it times the run
    if not (V / f'lib/pie/predictions/{line}.parquet').exists():
        bg(f'pie_{line}', f'bash scripts/pie_run.sh {line}'); wait(f'pie_{line}')
    if not (V / f'lib/pie_tables/pie_{line}.h5ad').exists():
        run(f'"$PY" scripts/pie_tables.py --lines {line}')
display(pd.DataFrame([json.loads((V / f'lib/pie/predictions/{l}_timing.json').read_text()) for l in PIE_LINES]))

### PIE on K562 (its clean test line) next to the cross-source baseline
Effect space, per knockdown then the median; truly DE = BH 5% in K562. Baseline = precision-weighted lfc of every other
source (T cells as Stim48hr, H1 included), score |z|; PIE = lfc_pred, score p_de.

In [ ]:
if not (V / 'results/analysis/k562_benchmark_pie.csv').exists():
    run('"$PY" scripts/k562_benchmark.py --pie lib/pie_tables/pie_K562.npz')
cols = ['r_all', 'r_de', 'sign_de', 'auprc', 'auprc_random']
rows = {}
for m in ('baseline', 'pie'):
    d = pd.read_csv(V / f'results/analysis/k562_benchmark_{m}.csv')
    both = set(pd.read_csv(V / 'results/analysis/k562_benchmark_pie.csv').target) & set(pd.read_csv(V / 'results/analysis/k562_benchmark_baseline.csv').target)
    d = d[d.target.isin(both)]
    rows[(m, 'all')] = d[cols].median(); rows[(m, 'n_de_bh>=50')] = d.loc[d.n_de_bh >= 50, cols].median()
display(pd.DataFrame(rows).T.round(3))

## Phase D: transformer build and smoke test

Code: `code/transformer/` (config, data, model, losses, train, evaluate), configs `code/transformer/configs/*.yaml`,
tests `tests/test_transformer.py`. Rows = (knockdown k, target line t); targets K562, RPE1, HepG2, Jurkat, HCT116,
HEK293T (validation mode) + H1 (final mode); CD4 T evidence only. Holdout 20 % of target genes (seeded); dev gene set =
our 300 targets + top genes to 5,000 by number of target lines measuring them, then mean |lfc/se|. Evidence per (gene,
source != t) + a PIE slot (off for HCT116 / HEK293T / H1); basal features from baseline v2; efficiency = median own-gene
lfc over evidence sources (+ N(0, 0.2) in training). Model: PIE gene features -> d = 128, evidence MLP + source
attention, 4-layer pre-norm TransformerEncoder (4 heads, FF 512, dropout 0.1, bf16); delta_hat = Linear + gamma x
precision-weighted evidence mean; p_de logit. Loss: weighted Huber + 0.1 (1 - cosine) + BCE (pos_weight <= 50).

*Result: filled in when the phase runs.*

In [ ]:
run('PYTHONPATH=code "$PY" -m pytest tests/test_transformer.py -q')

In [ ]:
if not (V / 'results/vcc_2026_4/smoke.json').exists():
    bg('smoke', 'PYTHONPATH=code "$PY" -m transformer.train smoke --config code/transformer/configs/tiny.yaml --out results/vcc_2026_4/smoke.json')
    wait('smoke')
sm = json.loads((V / 'results/vcc_2026_4/smoke.json').read_text())
display(pd.DataFrame({'overfit first': sm['overfit']['first'], 'overfit last': sm['overfit']['last']}).T)
print('200 real steps:', sm['real'])

In [ ]:
if not (V / 'results/vcc_2026_4/bench.json').exists():
    bg('bench', 'PYTHONPATH=code "$PY" -m transformer.train bench --config code/transformer/configs/dev.yaml --out results/vcc_2026_4/bench.json')
    wait('bench')
display(pd.DataFrame(json.loads((V / 'results/vcc_2026_4/bench.json').read_text())).round(3))

## Phase E: dev run and validation

*Filled in when the phase runs.*

## Phase F: submission

*Filled in when the phase runs.*